# **Setup**

* https://towardsdatascience.com/practical-sql-puzzles-that-will-level-up-your-skill/

In [25]:
import duckdb as db
import pandas as pd

# **DuckDB Connection**

In [26]:
db_conn = db.connect(database=':memory:', read_only=False)

# **Tickets Table**

In [27]:
db_conn.execute(
    ''' 
    CREATE TABLE ticket_moves (
    ticket_id INT NOT NULL,
    create_date DATE NOT NULL,
    move_date DATE NOT NULL,
    from_stage TEXT NOT NULL,
    to_stage TEXT NOT NULL
);
    '''
)

In [28]:
# Load the data
db_conn.execute( 
    '''  
    INSERT INTO ticket_moves (ticket_id, create_date, move_date, from_stage, to_stage)
    VALUES
        -- Ticket 1: Created in "New", then moves to Doing, Review, Done.
        (1, '2024-09-01', '2024-09-03', 'New', 'Doing'),
        (1, '2024-09-01', '2024-09-07', 'Doing', 'Review'),
        (1, '2024-09-01', '2024-09-10', 'Review', 'Done'),

        -- Ticket 2: Created in "New", then moves: New → Doing → Review → Doing again → Review.
        (2, '2024-09-05', '2024-09-08', 'New', 'Doing'),
        (2, '2024-09-05', '2024-09-12', 'Doing', 'Review'),
        (2, '2024-09-05', '2024-09-15', 'Review', 'Doing'),
        (2, '2024-09-05', '2024-09-20', 'Doing', 'Review'),
        
        -- Ticket 3: Created in "New", then moves to Doing. (Edge case: no subsequent move from Doing.)
        (3, '2024-09-10', '2024-09-16', 'New', 'Doing'),
        
        -- Ticket 4: Created already in "Doing", then moves to Review.
        (4, '2024-09-15', '2024-09-22', 'Doing', 'Review');
    '''
)

In [29]:
# Preview the table
db_conn.execute('SELECT * FROM ticket_moves').fetch_df()

,ticket_id,create_date,move_date,from_stage,to_stage
0,1,2024-09-01,2024-09-03,New,Doing
1,1,2024-09-01,2024-09-07,Doing,Review
2,1,2024-09-01,2024-09-10,Review,Done
3,2,2024-09-05,2024-09-08,New,Doing
4,2,2024-09-05,2024-09-12,Doing,Review
5,2,2024-09-05,2024-09-15,Review,Doing
6,2,2024-09-05,2024-09-20,Doing,Review
7,3,2024-09-10,2024-09-16,New,Doing
8,4,2024-09-15,2024-09-22,Doing,Review


## **Query 01**

In [30]:
# Average time in doing stage

db_conn.execute( 
    '''
    WITH stage_intervals AS (
        SELECT ticket_id, from_stage, move_date 
            - COALESCE( LAG(move_date) OVER (PARTITION BY ticket_id ORDER BY move_date), 
                create_date
            ) AS days_in_stage
        FROM ticket_moves) 
    
    SELECT SUM(days_in_stage) / COUNT(DISTINCT ticket_id) as avg_days_in_doing
    FROM stage_intervals
    WHERE from_stage = 'Doing'
    '''
).fetch_df()

,avg_days_in_doing
0,6.666667


# **Contract Table**

In [31]:
db_conn.execute( 
    '''  
    CREATE TABLE contracts (
    contract_id integer PRIMARY KEY,
    employee_id integer NOT NULL,
    start_date date NOT NULL,
    end_date date NOT NULL);

    INSERT INTO contracts (contract_id, employee_id, start_date, end_date)
    VALUES 
        -- Employee 1: Two continuous contracts
        (1, 1, '2024-01-01', '2024-03-31'),
        (2, 1, '2024-04-01', '2024-06-30'),

        -- Employee 2: One contract, then a gap of three days, then two contracts
        (3, 2, '2024-01-01', '2024-02-15'),
        (4, 2, '2024-02-19', '2024-04-30'),
        (5, 2, '2024-05-01', '2024-07-31'),

        -- Employee 3: One contract
        (6, 3, '2024-03-01', '2024-08-31');
        ''')

In [32]:
db_conn.execute('SELECT * FROM contracts').fetch_df()

,contract_id,employee_id,start_date,end_date
0,1,1,2024-01-01,2024-03-31
1,2,1,2024-04-01,2024-06-30
2,3,2,2024-01-01,2024-02-15
3,4,2,2024-02-19,2024-04-30
4,5,2,2024-05-01,2024-07-31
5,6,3,2024-03-01,2024-08-31


## **Query 02**

In [33]:
db_conn.execute(
    '''
    WITH 
    
    ordered_contracts AS (
                        SELECT *, 
                            LAG(end_date) OVER (PARTITION BY employee_id ORDER BY start_date) AS previous_end_date
                        FROM contracts),

    gapped_contracts AS (
        SELECT
            *,
            -- Deals with the case of the first contract, which won't have
            -- a previous end date. In this case, it's still the start of a new
            -- sequence.
            CASE WHEN previous_end_date IS NULL
                OR previous_end_date < start_date - INTERVAL '1 day' THEN
                1
            ELSE
                0
            END AS is_new_sequence
        FROM
            ordered_contracts),
    
    sequences AS (
        SELECT
            *, SUM(is_new_sequence) OVER (PARTITION BY employee_id ORDER BY start_date) AS sequence_id
        FROM gapped_contracts),
    
    max_sequence AS (
        SELECT employee_id, MAX(sequence_id) AS max_sequence_id
        FROM sequences
        GROUP BY employee_id),

latest_contract_sequence AS (
    SELECT c.contract_id, c.employee_id, c.start_date, c.end_date
    FROM sequences c
    JOIN max_sequence m ON c.sequence_id = m.max_sequence_id
            AND c.employee_id = m.employee_id
    ORDER BY c.employee_id, c.start_date)

SELECT * FROM latest_contract_sequence;
    '''
).fetch_df()

,contract_id,employee_id,start_date,end_date
0,1,1,2024-01-01,2024-03-31
1,2,1,2024-04-01,2024-06-30
2,4,2,2024-02-19,2024-04-30
3,5,2,2024-05-01,2024-07-31
4,6,3,2024-03-01,2024-08-31


# **Contract Time**

In [34]:
db_conn.execute( 
    '''   
    CREATE TABLE meetings (
    room TEXT NOT NULL,
    start_time TIMESTAMP NOT NULL,
    end_time TIMESTAMP NOT NULL
);

INSERT INTO meetings (room, start_time, end_time) VALUES
    -- Room A meetings
    ('Room A', '2024-10-01 09:00', '2024-10-01 10:00'),
    ('Room A', '2024-10-01 10:00', '2024-10-01 11:00'),
    ('Room A', '2024-10-01 11:00', '2024-10-01 12:00'),
    -- Room B meetings
    ('Room B', '2024-10-01 09:30', '2024-10-01 11:30'),
    -- Room C meetings
    ('Room C', '2024-10-01 09:00', '2024-10-01 10:00'),
    ('Room C', '2024-10-01 11:30', '2024-10-01 12:00');
'''
)

In [35]:
db_conn.execute('SELECT * FROM meetings').fetch_df()

,room,start_time,end_time
0,Room A,2024-10-01 09:00:00,2024-10-01 10:00:00
1,Room A,2024-10-01 10:00:00,2024-10-01 11:00:00
2,Room A,2024-10-01 11:00:00,2024-10-01 12:00:00
3,Room B,2024-10-01 09:30:00,2024-10-01 11:30:00
4,Room C,2024-10-01 09:00:00,2024-10-01 10:00:00
5,Room C,2024-10-01 11:30:00,2024-10-01 12:00:00


## **Query 03**

In [37]:
db_conn.execute(''' 
    WITH events AS (
        -- Create an event for the start of each meeting (+1)
        SELECT start_time AS event_time, 1 AS delta
        FROM meetings 
        UNION ALL 
        -- Create an event for the end of each meeting (-1)
        SELECT 
        -- Small trick to work with the back-to-back meetings (explained later)
            end_time - interval '1 minute' as end_time,
            -1 AS delta
        FROM meetings),
                
    ordered_events AS (
        SELECT event_time, delta, SUM(delta) OVER (ORDER BY event_time, delta DESC) AS concurrent_meetings
        FROM events),
    
    max_events AS (
        -- Find the maximum concurrent meetings value
        SELECT event_time, concurrent_meetings, RANK() OVER (ORDER BY concurrent_meetings DESC) AS rnk
        FROM ordered_events)
                
    SELECT event_time, concurrent_meetings
    FROM max_events
    WHERE rnk = 1;
''').fetch_df()

,event_time,concurrent_meetings
0,2024-10-01 09:30:00,3.0


# **Close DB Connection**

In [38]:
db_conn.close()